# CheXSpark — Stage-2: Multi-label Classification (DenseNet121)

Huấn luyện DenseNet121 dự đoán **14 bệnh lý lồng ngực** trên tập dev 2,000 ảnh NIH ChestX-ray14.

**Đính kèm 2 dataset vào notebook này (Add data):**
1. Ảnh 224×224: `khanfashee/nih-chest-x-ray-14-224x224-resized`
2. Nhãn + mask RLE: `<tai-khoan>/chexspark-stage2-nih-dev` (upload từ `data/kaggle/stage2-nih-dev`)

**Pipeline:** đọc ảnh → decode mask phổi CheXmask (RLE) → mask×ảnh (tắt nền ngoài phổi) → DenseNet121 → BCEWithLogitsLoss (pos_weight) → eval AUC/label.

Chọn **GPU** trước khi chạy (Accelerator → GPU T4 x2 / P100).

In [ ]:
# ============ CONFIG — chỉnh ở đây nếu path Kaggle khác ============
import os, glob


def resolve_input(ref):
    """Tìm thư mục dataset đã mount.

    Kaggle mới (2025+) mount tại /kaggle/input/datasets/<owner>/<slug>;
    bản cũ tại /kaggle/input/<slug>. Dò lần lượt, cuối cùng là glob.
    """
    if not ref:
        return ''
    slug = ref.split('/')[-1]
    for cand in (f'/kaggle/input/datasets/{ref}', f'/kaggle/input/{ref}',
                 f'/kaggle/input/{slug}'):
        if os.path.isdir(cand):
            return cand
    hits = [h for h in sorted(glob.glob(f'/kaggle/input/**/{slug}', recursive=True))
            if os.path.isdir(h)]
    return hits[0] if hits else f'/kaggle/input/{slug}'


IMG_DS   = 'khanfashee/nih-chest-x-ray-14-224x224-resized'   # ảnh 224x224 PNG
META_DS  = 'pnt2005/chexspark-stage2-nih-dev'               # stage2_train.csv
MASKS_DS = ''                                               # mask pack (chexspark-masks-224)

IMG_ROOT      = resolve_input(IMG_DS)
META_ROOT     = resolve_input(META_DS) if META_DS else ''
MASK_PACK_DIR = resolve_input(MASKS_DS) if MASKS_DS else ''
TRAIN_CSV     = os.path.join(META_ROOT, 'stage2_train.csv')

IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 8
LR          = 1e-4
NUM_WORKERS = 4

# Cách đưa mask vào ảnh:
#   '3ch'    = [ảnh gốc, ảnh*phổi, ảnh*(phổi|tim)] — giữ ngữ cảnh trung thất/tim
#   'masked' = ảnh*phổi (cách cũ: xoá sạch tim -> hại Cardiomegaly)
#   'none'   = không dùng mask (đối chứng)
MASK_INPUT  = '3ch'
MASK_MODE   = 'lungs_heart'    # chỉ dùng khi fallback decode RLE từ CSV

# Augmentation (áp CÙNG một tham số cho ảnh và mask)
AUG       = True
AUG_SCALE = 0.85               # RandomResizedCrop: scale thấp nhất
AUG_ROT   = 7.0                # xoay ± độ

TTA        = True              # trung bình x và hflip(x) lúc eval
CALIBRATE  = True              # temperature + threshold theo từng nhãn (fit trên val)

# Resume: trỏ tới 'last_stage2_densenet121.pth' của lần chạy trước
RESUME_FROM   = ''

USE_AMP     = True
SEED        = 42

print('paths |', IMG_ROOT, '|', META_ROOT, '|', MASK_PACK_DIR or '(no mask pack)')


In [ ]:
import os, glob, math, random, shutil, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| torch', torch.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# torch >= 2.4 chuyen sang torch.amp; giu fallback cho ban cu
try:
    from torch import amp as _amp
    def autocast(enabled=True):
        return _amp.autocast('cuda', enabled=enabled)
except Exception:                                    # torch < 2.4
    def autocast(enabled=True):
        return torch.cuda.amp.autocast(enabled=enabled)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
set_seed(SEED)


In [ ]:
# Xây index {basename: full_path} để join theo image_index (không đoán cấu trúc folder)
print('Quét ảnh trong', IMG_ROOT, '...')
img_index = {}
for dirpath, _, filenames in os.walk(IMG_ROOT):
    for fn in filenames:
        if fn.lower().endswith('.png'):
            img_index[fn] = os.path.join(dirpath, fn)
print(f'Tìm thấy {len(img_index)} PNG')
if len(img_index) == 0:
    print('-> Không thấy PNG loose. Top-level:', os.listdir(IMG_ROOT))
    print('   (nếu ảnh nằm trong .zip, cần giải nén — chỉnh cell này)')

In [ ]:
df = pd.read_csv(TRAIN_CSV, dtype={'image_index': str})
df['path'] = df['image_index'].map(img_index)
found = df['path'].notna().sum()
print(f'Rows: {len(df)} | có ảnh trên Kaggle: {found} ({found/len(df):.1%})')
if found < len(df):
    print('Thiếu (ví dụ):', df.loc[df['path'].isna(), 'image_index'].head(5).tolist())
df = df.dropna(subset=['path']).reset_index(drop=True)

LABELS = [c for c in df.columns if c.startswith('label_')]
print('Labels:', len(LABELS), LABELS)

tr_df = df[df.split == 'train'].reset_index(drop=True)
va_df = df[df.split == 'val'].reset_index(drop=True)
te_df = df[df.split == 'test'].reset_index(drop=True)
print(f'train/val/test = {len(tr_df)}/{len(va_df)}/{len(te_df)}')

# ---- mask pack: 1 memmap read + unpackbits (~0.1ms) thay decode RLE (~8-12ms) ----
NEED_HEART = (MASK_INPUT == '3ch') or (MASK_MODE == 'lungs_heart')
mask_pack, mask_heart_pack, mask_pos = None, None, {}

if MASK_INPUT == 'none':
    print('mask: MASK_INPUT=none -> không dùng mask (đối chứng)')
elif MASK_PACK_DIR and os.path.isdir(MASK_PACK_DIR):
    _mi = pd.read_csv(os.path.join(MASK_PACK_DIR, 'mask_index.csv'),
                      dtype={'image_index': str})
    mask_pos = {(r.split, r.image_index): int(r.mask_row) for r in _mi.itertuples(index=False)}
    mask_pack = {s: np.load(os.path.join(MASK_PACK_DIR, f'masks_{s}.npy'), mmap_mode='r')
                 for s in ('train', 'val', 'test')}
    _hp = os.path.join(MASK_PACK_DIR, 'masks_heart_train.npy')
    if os.path.exists(_hp):
        mask_heart_pack = {s: np.load(os.path.join(MASK_PACK_DIR, f'masks_heart_{s}.npy'),
                                      mmap_mode='r') for s in ('train', 'val', 'test')}
    _miss = sum(1 for s, i in zip(df.split, df.image_index) if (s, i) not in mask_pos)
    print(f'mask pack: {MASK_PACK_DIR} | thiếu {_miss}/{len(df)} | '
          f'heart={mask_heart_pack is not None}')
    if NEED_HEART and mask_heart_pack is None:
        print('[WARN] thiếu masks_heart_*.npy -> kênh 3 trùng kênh 2')
else:
    print('mask pack: KHÔNG dùng -> decode RLE từ CSV (chậm hơn ~100x)')


In [ ]:
def decode_rle(rle, h, w):
    """CheXmask RLE: chuoi cap (start, length), reshape row-major (C order)."""
    vals = [int(v) for v in str(rle).split()]
    flat = np.zeros(h * w, dtype=np.uint8)
    for i in range(0, len(vals), 2):
        flat[vals[i]: vals[i] + vals[i + 1]] = 1
    return flat.reshape((h, w))


def _resize_mask(m, size):
    if m.shape == (size, size):
        return m.astype(np.float32)
    img = Image.fromarray((m * 255).astype(np.uint8)).resize((size, size), Image.NEAREST)
    return (np.array(img) > 127).astype(np.float32)


def _crop_params(h, w, scale_min, ratio=(0.9, 1.11), tries=10):
    """Tham so RandomResizedCrop tu tinh (khong phu thuoc API torchvision)."""
    area = h * w
    for _ in range(tries):
        target = area * random.uniform(scale_min, 1.0)
        ar = math.exp(random.uniform(math.log(ratio[0]), math.log(ratio[1])))
        cw = int(round(math.sqrt(target * ar)))
        ch = int(round(math.sqrt(target / ar)))
        if 0 < cw <= w and 0 < ch <= h:
            return random.randint(0, h - ch), random.randint(0, w - cw), ch, cw
    return 0, 0, h, w


def augment_pair(g, lung, heart):
    """Ap CUNG mot phep bien doi hinh hoc len anh + mask: tensor (1,H,W).

    RRC(scale >= AUG_SCALE) + hflip + rotate +-AUG_ROT. Khong vflip (vo nghia giai phau).
    """
    h, w = g.shape[-2], g.shape[-1]
    i, j, ch, cw = _crop_params(h, w, AUG_SCALE)
    g     = TF.resized_crop(g,     i, j, ch, cw, [h, w], TF.InterpolationMode.BILINEAR)
    lung  = TF.resized_crop(lung,  i, j, ch, cw, [h, w], TF.InterpolationMode.NEAREST)
    heart = TF.resized_crop(heart, i, j, ch, cw, [h, w], TF.InterpolationMode.NEAREST)

    if random.random() < 0.5:
        g, lung, heart = TF.hflip(g), TF.hflip(lung), TF.hflip(heart)

    ang = random.uniform(-AUG_ROT, AUG_ROT)
    if abs(ang) > 0.05:
        g     = TF.rotate(g,     ang, interpolation=TF.InterpolationMode.BILINEAR, fill=0)
        lung  = TF.rotate(lung,  ang, interpolation=TF.InterpolationMode.NEAREST,  fill=0)
        heart = TF.rotate(heart, ang, interpolation=TF.InterpolationMode.NEAREST,  fill=0)

    return g, (lung > 0.5).float(), (heart > 0.5).float()


class CheXSparkDataset(Dataset):
    def __init__(self, frame, img_size=224, mask_mode='lungs_heart',
                 mask_input='3ch', aug=False):
        self.df = frame
        self.img_size = img_size
        self.mask_mode = mask_mode
        self.mask_input = mask_input
        self.aug = aug
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def _masks(self, row):
        """Tra ve (lung, heart) float32 cung kich thuoc anh."""
        # 1) mask precompute: memmap read + unpackbits (~0.1ms)
        if mask_pack is not None and self.img_size == IMG_SIZE:
            pos = mask_pos.get((row['split'], row['image_index']))
            if pos is not None:
                m = np.unpackbits(mask_pack[row['split']][pos]).reshape(IMG_SIZE, IMG_SIZE)
                if mask_heart_pack is not None:
                    hm = np.unpackbits(mask_heart_pack[row['split']][pos]).reshape(
                        IMG_SIZE, IMG_SIZE)
                else:
                    hm = np.zeros_like(m)
                return m.astype(np.float32), hm.astype(np.float32)

        # 2) fallback: decode RLE 1024x1024 (~8-12ms/anh)
        h, w = int(row['mask_height']), int(row['mask_width'])
        m = decode_rle(row['mask_left_rle'], h, w) | decode_rle(row['mask_right_rle'], h, w)
        hm = decode_rle(row['mask_heart_rle'], h, w)
        return _resize_mask(m, self.img_size), _resize_mask(hm, self.img_size)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('L').resize((self.img_size, self.img_size))
        g = torch.from_numpy(np.array(img).astype(np.float32) / 255.0)[None]

        lung = heart = None
        if self.mask_input != 'none':
            lm, hm = self._masks(row)
            lung = torch.from_numpy(lm)[None]
            heart = torch.from_numpy(hm)[None]
            if self.aug:
                g, lung, heart = augment_pair(g, lung, heart)

        if self.mask_input == '3ch':
            # kenh 0 = anh goc (giu trung that/tim), kenh 1 = vung phoi, kenh 2 = phoi+tim
            x = torch.cat([g, g * lung, g * torch.clamp(lung + heart, 0, 1)], dim=0)
        elif self.mask_input == 'masked':
            x = torch.cat([g * lung] * 3, dim=0)       # cach cu: nhan cung
        else:
            x = torch.cat([g] * 3, dim=0)              # doi chung: khong mask

        x = self.norm(x)
        y = torch.tensor([row[c] for c in LABELS], dtype=torch.float32)
        return x, y


In [ ]:
# Kiem tra nhanh: decode 1 mask + overlay len anh (doi chieu train/val/test)
row = te_df.iloc[0]
h, w = int(row['mask_height']), int(row['mask_width'])
lung = decode_rle(row['mask_left_rle'], h, w) | decode_rle(row['mask_right_rle'], h, w)
img = np.array(Image.open(row['path']).convert('L').resize((IMG_SIZE, IMG_SIZE)))
m = np.array(Image.fromarray((lung * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) > 127
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img, cmap='gray'); ax[0].set_title(row['image_index']); ax[0].axis('off')
ax[1].imshow(m, cmap='gray');   ax[1].set_title('lung mask');       ax[1].axis('off')
ax[2].imshow(img * m, cmap='gray'); ax[2].set_title('masked');      ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# pos_weight = N_neg / N_pos per label (tu train), clamp tranh cuc dai
pos = tr_df[LABELS].sum().clip(lower=1)
neg = len(tr_df) - tr_df[LABELS].sum()
pos_weight = torch.tensor((neg / pos).clip(upper=20).values, dtype=torch.float32).to(device)
print('pos_weight (5 dau):', [round(v, 2) for v in pos_weight[:5].tolist()])

train_ds = CheXSparkDataset(tr_df, IMG_SIZE, MASK_MODE, MASK_INPUT, aug=AUG)
val_ds   = CheXSparkDataset(va_df, IMG_SIZE, MASK_MODE, MASK_INPUT, aug=False)
test_ds  = CheXSparkDataset(te_df, IMG_SIZE, MASK_MODE, MASK_INPUT, aug=False)

_g = torch.Generator(); _g.manual_seed(SEED)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, generator=_g, drop_last=True,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=NUM_WORKERS > 0)
val_dl   = DataLoader(val_ds, BATCH_SIZE * 2, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)
test_dl  = DataLoader(test_ds, BATCH_SIZE * 2, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)
print(f'DataLoaders ready | mask_input={MASK_INPUT} aug={AUG} tta={TTA} | '
      f'train/val/test = {len(train_ds)}/{len(val_ds)}/{len(test_ds)}')


In [ ]:
def build_model(n_classes):
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, n_classes)
    return m

model = build_model(len(LABELS)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == 'cuda')


@torch.no_grad()
def predict_logits(dl, tta=False):
    model.eval()
    Ys, Zs = [], []
    for x, y in dl:
        x = x.to(device, non_blocking=True)
        with autocast(scaler.is_enabled()):
            z = model(x)
            if tta:                                   # TTA: trung binh voi anh lat ngang
                z = 0.5 * (z + model(torch.flip(x, dims=[3])))
        Ys.append(y.numpy().astype(np.float32))
        Zs.append(z.float().cpu().numpy())
    return np.concatenate(Ys), np.concatenate(Zs)


def auc_summary(Y, P):
    aucs = []
    for i in range(Y.shape[1]):
        if 0 < Y[:, i].sum() < len(Y):
            aucs.append(roc_auc_score(Y[:, i], P[:, i]))
        else:
            aucs.append(float('nan'))                 # nhan degenerate trong tap nay
    return float(np.nanmean(aucs)), aucs


def evaluate(dl, tta=False):
    Y, Z = predict_logits(dl, tta=tta)
    P = 1.0 / (1.0 + np.exp(-Z))
    m, per = auc_summary(Y, P)
    return m, per, Y, Z


def fit_temperature(Y, Z):
    """Temperature scaling tung nhan (grid search, khong can LBFGS)."""
    grid = np.linspace(0.5, 4.0, 36)
    out = np.ones(Y.shape[1], dtype=np.float32)
    for i in range(Y.shape[1]):
        y, z = Y[:, i], Z[:, i]
        if not (0 < y.sum() < len(y)):
            continue
        best, best_t = None, 1.0
        for t in grid:
            p = np.clip(1.0 / (1.0 + np.exp(-z / t)), 1e-6, 1 - 1e-6)
            bce = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
            if best is None or bce < best:
                best, best_t = bce, float(t)
        out[i] = best_t
    return out


def pick_thresholds(Y, P):
    """Nguong toi da hoa Youden J = sens + spec - 1, fit tren val."""
    th = np.full(Y.shape[1], 0.5, dtype=np.float32)
    grid = np.linspace(0.05, 0.95, 91)
    for i in range(Y.shape[1]):
        y = Y[:, i]
        npos, nneg = float(y.sum()), float((1 - y).sum())
        if not (npos > 0 and nneg > 0):
            continue
        best, best_t = -1.0, 0.5
        for t in grid:
            pred = P[:, i] >= t
            sens = (pred & (y > 0.5)).sum() / npos
            spec = 1 - (pred & (y <= 0.5)).sum() / nneg
            if sens + spec - 1 > best:
                best, best_t = sens + spec - 1, float(t)
        th[i] = best_t
    return th


def sens_spec(y, p, t):
    pred = p >= t
    npos, nneg = float((y > 0.5).sum()), float((y <= 0.5).sum())
    se = float((pred & (y > 0.5)).sum() / npos) if npos else float('nan')
    sp = float(1 - (pred & (y <= 0.5)).sum() / nneg) if nneg else float('nan')
    return se, sp


print('Model + loss + eval ready.')

# ---- Resume (Kaggle cat session 12h -> chay tiep tu checkpoint) ------------ #
START_EPOCH, best = 1, 0.0
history = {'train_loss': [], 'val_auc': []}
if RESUME_FROM:
    ckpt_path = RESUME_FROM
    if not os.path.exists(ckpt_path):          # layout Kaggle đổi -> tự dò
        hits = glob.glob('/kaggle/input/**/last_stage2_densenet121*.pth', recursive=True)
        ckpt_path = max(hits, key=os.path.getmtime) if hits else ''
    if os.path.isdir(ckpt_path):
        cands = sorted(glob.glob(os.path.join(ckpt_path, 'last_stage2_densenet121*.pth')),
                       key=os.path.getmtime)
        ckpt_path = cands[-1] if cands else ''
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        if scaler.is_enabled() and ck.get('scaler'):
            scaler.load_state_dict(ck['scaler'])
        START_EPOCH = int(ck['epoch']) + 1
        best = float(ck.get('best', 0.0))
        history = ck.get('history', history)
        prev_best = os.path.join(os.path.dirname(ckpt_path), 'best_stage2_densenet121.pth')
        if os.path.exists(prev_best) and os.path.abspath(prev_best) != os.path.abspath(
                'best_stage2_densenet121.pth'):
            shutil.copy2(prev_best, 'best_stage2_densenet121.pth')
        print(f'Resume tu {ckpt_path}: epoch {START_EPOCH}/{EPOCHS}, best_val_mAU={best:.4f}')
    else:
        print(f'Khong thay checkpoint tai {RESUME_FROM!r} -> train tu dau')
else:
    print(f'Train tu dau: 1..{EPOCHS}')


In [ ]:
last_epoch = START_EPOCH - 1
for ep in range(START_EPOCH - 1, EPOCHS):
    model.train()
    t0, tot, seen = time.time(), 0.0, 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(scaler.is_enabled()):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        tot += loss.item() * len(x)
        seen += len(x)
    scheduler.step()
    tr_loss = tot / max(seen, 1)
    va_auc, _, _, _ = evaluate(val_dl)          # chon checkpoint bang val (khong TTA)
    history['train_loss'].append(tr_loss); history['val_auc'].append(va_auc)
    print(f'Ep{ep+1:02d}/{EPOCHS}  loss={tr_loss:.4f}  val_mAU={va_auc:.4f}  '
          f'({time.time()-t0:.0f}s)')
    if va_auc > best:
        best = va_auc
        torch.save(model.state_dict(), 'best_stage2_densenet121.pth')
        print(f'  -> checkpoint moi (val_mAU={best:.4f})')
    last_epoch = ep + 1
    # Luu ca trang thai optimizer de resume sau khi Kaggle cat session 12h
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict() if scaler.is_enabled() else {},
                'epoch': ep + 1, 'best': best, 'history': history},
               'last_stage2_densenet121.pth')
    with open('stage2_metrics.json', 'w') as f:
        json.dump({'best_val_auc': best, 'last_epoch': ep + 1, 'history': history,
                   'resumed_from': RESUME_FROM or None,
                   'config': {'img_size': IMG_SIZE, 'epochs': EPOCHS, 'lr': LR,
                              'mask_mode': MASK_MODE, 'mask_input': MASK_INPUT,
                              'aug': bool(AUG), 'tta': bool(TTA),
                              'mask_pack': mask_pack is not None,
                              'amp': scaler.is_enabled()}}, f, indent=2)
print(f'Best val mAU: {best:.4f} | da train toi epoch {last_epoch}/{EPOCHS}')


In [ ]:
if os.path.exists('best_stage2_densenet121.pth'):
    model.load_state_dict(torch.load('best_stage2_densenet121.pth', map_location=device))
else:
    print('[WARN] khong co best_stage2_densenet121.pth -> dung trong so cuoi')

val_mau, _, Yv, Zv = evaluate(val_dl, tta=TTA)
test_mau, test_aucs, Yt, Zt = evaluate(test_dl, tta=TTA)
print(f'VAL  mean AUC = {val_mau:.4f}')
print(f'TEST mean AUC = {test_mau:.4f}  (TTA={TTA}, mask_input={MASK_INPUT})')

# ---- Calibration: temperature (lam mem xac suat) + threshold tung nhan ----
temperature = np.ones(len(LABELS), dtype=np.float32)
thresholds = np.full(len(LABELS), 0.5, dtype=np.float32)
if CALIBRATE:
    temperature = fit_temperature(Yv, Zv)                    # fit tren VAL
    Pv = 1.0 / (1.0 + np.exp(-Zv / temperature[None, :]))
    thresholds = pick_thresholds(Yv, Pv)
    print('Da fit temperature + threshold tren val')

Pt = 1.0 / (1.0 + np.exp(-Zt / temperature[None, :]))
rows = []
print('\n%-22s %6s %5s %6s %6s' % ('label', 'AUC', 'thr', 'sens', 'spec'))
for i, name in enumerate(LABELS):
    short = name.replace('label_', '')
    a = test_aucs[i]
    se, sp = sens_spec(Yt[:, i], Pt[:, i], float(thresholds[i]))
    auc_s = 'nan' if pd.isna(a) else '%.3f' % a
    se_s = 'nan' if pd.isna(se) else '%.3f' % se
    sp_s = 'nan' if pd.isna(sp) else '%.3f' % sp
    rows.append({'label': short,
                 'auc': None if pd.isna(a) else float(a),
                 'threshold': float(thresholds[i]),
                 'sensitivity': None if pd.isna(se) else se,
                 'specificity': None if pd.isna(sp) else sp})
    print('%-22s %6s %5.2f %6s %6s' % (short, auc_s, thresholds[i], se_s, sp_s))


In [ ]:
# Luu model + metrics (working dir cua Kaggle Notebook Output)
torch.save(model.state_dict(), 'stage2_densenet121_final.pth')
with open('stage2_metrics.json', 'w') as f:
    json.dump({
        'best_val_auc': best,
        'last_epoch': last_epoch,
        'resumed_from': RESUME_FROM or None,
        'test_auc': test_mau,                     # giu key cu cho registry
        'per_label_auc': {r['label']: r['auc'] for r in rows},
        'test': {'mauroc': test_mau, 'val_mauroc': val_mau, 'tta': bool(TTA)},
        'calibration': {
            'method': 'temperature_scaling + youden_threshold',
            'temperature': [float(t) for t in temperature],
            'threshold': [float(t) for t in thresholds],
            'per_label': rows,
        },
        'config': {'img_size': IMG_SIZE, 'epochs': EPOCHS, 'lr': LR,
                   'mask_mode': MASK_MODE, 'mask_input': MASK_INPUT,
                   'aug': bool(AUG), 'tta': bool(TTA),
                   'mask_pack': bool(MASK_PACK_DIR), 'amp': USE_AMP},
    }, f, indent=2)
print('Da luu stage2_densenet121_final.pth + stage2_metrics.json.')
print('De giu lai: xem Kernel Output, hoac push thanh Kaggle Dataset.')
